In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time
import matplotlib.pyplot as plt
from matplotlib import cm
from collections import OrderedDict
import scipy.io

In [ ]:
# Set random seeds for reproducibility
np.random.seed(0)
torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
Lx = 5.0     # Spatial domain: x in [0, Lx]
Ly = 5.0     # Spatial domain: y in [0, Ly]
T = 0.1      # Temporal domain: t in [0, T] 
c = 1.0      # Wave speed
m = 3      # Mode number in x-direction
n = 1       # Mode number in y-direction

nt = 10    # Number of time steps for collocation points
mesh_cases = [(26, 26, 3, 1), (34, 34, 3, 1), (51, 51, 3, 1)]

nx = 26    # Number of spatial points in x-direction for collocation
ny = 26    # Number of spatial points in y-direction for collocation
n_collocation = nt * nx * ny  # Number of collocation points inside domain
n_boundary    = 500   # Number of points for boundary conditions (each boundary)
n_initial     = nx*ny  # Number of points for initial conditions
hidden_layers = 6     # Number of hidden layers 
neurons       = 100 #50    # Neurons per hidden layer
learning_rate = 1e-3  # Learning rate for optimizer
num_epochs    = 5000 # 4000 # Training epochs

fourier_mode = 'spacetime'

log_NTK = True
update_lam  = True   # update loss weights using NTK
kernel_size = 400

def set_mesh_case(nx_value, ny_value, m_value=3, n_value=1):
    global nx, ny, m, n, n_collocation, n_initial
    nx, ny, m, n = nx_value, ny_value, m_value, n_value
    n_collocation = nt * nx * ny
    n_initial = nx * ny
    return f'mesh_{nx}_{ny}_{m}_{n}'

current_mesh_tag = set_mesh_case(*mesh_cases[0])


In [ ]:
class DNN(torch.nn.Module):
    def __init__(self, layers):
        super(DNN, self).__init__()

        # parameters
        self.depth = len(layers) - 1

        # set up layer order dict
        self.activation = torch.nn.Tanh

        layer_list = list()
        for i in range(self.depth - 1):
            layer_list.append(
                ('layer_%d' % i, torch.nn.Linear(layers[i], layers[i + 1]))
            )
            layer_list.append(('activation_%d' % i, self.activation()))

        layer_list.append(
            ('layer_%d' % (self.depth - 1), torch.nn.Linear(layers[-2], layers[-1]))
        )
        layerDict = OrderedDict(layer_list)

        # deploy layers
        self.layers = torch.nn.Sequential(layerDict)

    def forward(self, x):
        out = self.layers(x)
        return out
    

class sampling_MMPDE_initial():
    def __init__(self, X, Y, u_fun, layers, lb, ub, c, dt, nu, AdamIter, LBFGSIter):
        self.lb = torch.tensor(lb).float().to(device)
        self.ub = torch.tensor(ub).float().to(device)

        #self.t_f = torch.tensor(X_f[:, 0:1], requires_grad=True).float().to(device)
        self.x_f = torch.tensor(X, requires_grad=True).float().to(device)
        self.y_f = torch.tensor(Y, requires_grad=True).float().to(device)
        self.fun = u_fun

        self.layers = layers
        self.nu = nu
        self.c = c
        self.dt = dt

        self.dnn = DNN(layers).to(device)

        self.optimizer_Adam = torch.optim.Adam(self.dnn.parameters(),
                                              lr=1e-3,
                                              betas=(0.9, 0.999),
                                              eps=1e-8)
        
        self.AdamIter = AdamIter

        self.optimizer_LBFGS = torch.optim.LBFGS(
            self.dnn.parameters(),
            lr=0.5,
            max_iter=LBFGSIter,
        )
        
        self.optimizer = None
        self.loss = None
        self.iter = 0
        self.start_time = None

    def detach(self, data):
        return data.detach().cpu().numpy()

    def monitor(self, x, y):

        #t.requires_grad_(True)
        x.requires_grad_(True) 
        y.requires_grad_(True)

        # f1_tensor expects (x, y); model 等接受 cat 后的 tensor
        u = self.fun(x, y)
        #u = uv[:, [0]]  
        #v = uv[:, [1]]  

        u_x = torch.autograd.grad(
            u, x,
            grad_outputs=torch.ones_like(u),
            retain_graph=True,
            create_graph=True,
            allow_unused=True
        )[0]
        
        u_y = torch.autograd.grad(
            u, y,
            grad_outputs=torch.ones_like(u),
            retain_graph=True,
            create_graph=True,
            allow_unused=True
        )[0]



        #w = (1 + (u_x)**2 + (u_y)**2 + 0.1 * (v_t)**2) ** (1/2)
        w =  ( 0.01*((u_x)**2 + (u_y)**2)  + self.c ) ** (1/2)
        #print(f"监控函数 - w: {w.mean().item():.6e}")
        
        return w

    def net_sample(self, x, y):
        inputs = torch.cat([x, y], dim=1)
        xy_new = self.dnn(inputs)

        x_new = xy_new[:, [0]]
        y_new = xy_new[:, [1]]

        gx0 = x - self.lb[0] 
        gx1 = x - self.ub[0] 
        gy0 = y - self.lb[1] 
        gy1 = y - self.ub[1]

        #x_new = gx0 * gx1 * x_new + x
        #y_new = gy0 * gy1 * y_new + y
        
        return x_new, y_new

    def net_f(self, x, y):
        x_new, y_new = self.net_sample(x, y)


        x_new_x = torch.autograd.grad(
            x_new, x,
            grad_outputs=torch.ones_like(x_new),
            retain_graph=True,
            create_graph=True
        )[0]

        x_new_y = torch.autograd.grad(
            x_new, y,
            grad_outputs=torch.ones_like(x_new),
            retain_graph=True,
            create_graph=True
        )[0]

        y_new_x = torch.autograd.grad(
            y_new, x,
            grad_outputs=torch.ones_like(y_new),
            retain_graph=True,
            create_graph=True
        )[0]

        y_new_y = torch.autograd.grad(
            y_new, y,
            grad_outputs=torch.ones_like(y_new),
            retain_graph=True,
            create_graph=True
        )[0]

        x_new_xx = torch.autograd.grad(
            x_new_x, x,
            grad_outputs=torch.ones_like(x_new_x),
            retain_graph=True,
            create_graph=True
        )[0]

        x_new_yy = torch.autograd.grad(
            x_new_y, y,
            grad_outputs=torch.ones_like(x_new_y),
            retain_graph=True,
            create_graph=True
        )[0]

        y_new_xx = torch.autograd.grad(
            y_new_x, x,
            grad_outputs=torch.ones_like(y_new_x),
            retain_graph=True,
            create_graph=True
        )[0]

        y_new_yy = torch.autograd.grad(
            y_new_y, y,
            grad_outputs=torch.ones_like(y_new_y),
            retain_graph=True,
            create_graph=True
        )[0]

        G = self.monitor(x, y)
        print(f"监控函数 - w: {G.mean().item():.6e}")

        G_x = torch.autograd.grad(
            G, x,
            grad_outputs=torch.ones_like(G),
            retain_graph=True,
            create_graph=True
        )[0]

        G_y = torch.autograd.grad(
            G, y,
            grad_outputs=torch.ones_like(G),
            retain_graph=True,
            create_graph=True
        )[0]
        
        J = x_new_x * y_new_y - x_new_y * y_new_x  # Jocabi
        A = (x_new_y**2 + y_new_y**2) / (G*J)
        B = (x_new_x * x_new_y + y_new_x * y_new_y) / (G*J)
        C = (x_new_x**2 + y_new_x**2) / (G*J)
        
        A_x = torch.autograd.grad(
            A, x,
            grad_outputs=torch.ones_like(A),
            retain_graph=True,
            create_graph=True
        )[0]
        
        
        B_x = torch.autograd.grad(
            B, x,
            grad_outputs=torch.ones_like(B),
            retain_graph=True,
            create_graph=True
        )[0]
        
        B_y = torch.autograd.grad(
            B, x,
            grad_outputs=torch.ones_like(B),
            retain_graph=True,
            create_graph=True
        )[0]
        
        C_y = torch.autograd.grad(
            C, y,
            grad_outputs=torch.ones_like(C),
            retain_graph=True,
            create_graph=True
        )[0]
        
        
        # For x coordinate:
        #E_x = G_x * x_new_x + G_y * x_new_y + G * (x_new_xx + x_new_yy)
        #f_x = x_new_t * self.nu * (G**2) * (x_new_x**2 + x_new_y**2) + E_x
        f_x = ( ((x_new - x)/self.dt) * self.nu * J 
               + x_new_x * (A_x-B_y)
               + x_new_y * (C_y-B_x))
        
        
        # For y coordinate:
        #E_y = G_x * y_new_x + G_y * y_new_y + G * (y_new_xx + y_new_yy)
        #f_y = y_new_t * self.nu * (G**2) * (y_new_x**2 + y_new_y**2) + E_y
        f_y = (((y_new - y)/self.dt)  * self.nu * J 
               + y_new_x * (A_x-B_y)
               + y_new_y * (C_y-B_x))
        
        return f_x, f_y

    def loss_func(self):
        f_x, f_y = self.net_f(self.x_f, self.y_f)
        loss_f = torch.mean(f_x ** 2) + torch.mean(f_y ** 2)
        
        return loss_f

    def optimize_one_epoch(self):
        if self.start_time is None:
            self.start_time = time.time()

        self.optimizer.zero_grad()
        self.loss = self.loss_func()
        self.loss.backward()
        self.iter = self.iter + 1

        if self.iter % 100 == 0:
            loss = self.detach(self.loss)
            print(f"{self.optimizer_name} Iter {self.iter}, Loss: {loss:.6f}")
            elapsed = time.time() - self.start_time
            print(f'Time: {elapsed:.4f}s')
            self.start_time = time.time()

        return self.loss

    def train_Adam(self, optimizer, nIter):
        self.optimizer = optimizer
        self.optimizer_name = '2D_EMMPDE_Adam'
        
        for it in range(nIter):
            self.optimize_one_epoch()
            self.optimizer.step()

    def train_LBFGS(self, optimizer):
        self.optimizer = optimizer
        self.optimizer_name = '2D_EMMPDE_LBFGS'

        def closure():
            loss = self.optimize_one_epoch()
            return loss

        self.optimizer.step(closure)
    
    def train(self):
        print("开始2D initial points resampling MMPDE training...")
        self.train_Adam(self.optimizer_Adam, self.AdamIter)
        print("2D initial points resampling MMPDE training_Adam Done!")
        self.train_LBFGS(self.optimizer_LBFGS)
        print('2D initial points resampling MMPDE_LBFGS Done!')

        # 返回新的采样点
        x_new, y_new = self.net_sample(self.x_f, self.y_f)
        new_sample = torch.cat([x_new, y_new], dim=1)
        return new_sample
    


class sampling_MMPDE_initial_new():
    """
    在 sampling_MMPDE_initial 基础上修改：
    - 输入：X_f [N,3]，列依次为 (t, x, y) in [0,T]*[0,Lx]*[0,Ly]
    - monitor function：f1_tensor(x, y)，与时间无关
    - 使用 autograd 计算真实时间导数 x_new_t, y_new_t
    - 输出：t=T 时刻的采样点 [N,2] (x_new, y_new)
    """
    def __init__(self, X_f, u_fun, layers, lb, ub, c, nu, AdamIter, LBFGSIter):
        self.lb = torch.tensor(lb).float().to(device)
        self.ub = torch.tensor(ub).float().to(device)

        self.t_f = torch.tensor(X_f[:, 0:1], requires_grad=True).float().to(device)
        self.x_f = torch.tensor(X_f[:, 1:2], requires_grad=True).float().to(device)
        self.y_f = torch.tensor(X_f[:, 2:3], requires_grad=True).float().to(device)
        self.fun = u_fun  # f1_tensor(x, y), time-independent monitor

        self.layers = layers
        self.nu = nu
        self.c = c

        self.dnn = DNN(layers).to(device)

        self.optimizer_Adam = torch.optim.Adam(self.dnn.parameters(),
                                              lr=1e-3,
                                              betas=(0.9, 0.999),
                                              eps=1e-8)
        self.AdamIter = AdamIter

        self.optimizer_LBFGS = torch.optim.LBFGS(
            self.dnn.parameters(),
            lr=0.5,
            max_iter=LBFGSIter,
        )

        self.optimizer = None
        self.loss = None
        self.iter = 0
        self.start_time = None

    def detach(self, data):
        return data.detach().cpu().numpy()

    def monitor(self, x, y):
        """与时间无关的 monitor function G = (u_x^2 + u_y^2 + c) ^ (1/2)"""
        x.requires_grad_(True)
        y.requires_grad_(True)
        u = self.fun(x, y)
        u_x = torch.autograd.grad(
            u, x, grad_outputs=torch.ones_like(u),
            retain_graph=True, create_graph=True, allow_unused=True
        )[0]
        u_y = torch.autograd.grad(
            u, y, grad_outputs=torch.ones_like(u),
            retain_graph=True, create_graph=True, allow_unused=True
        )[0]
        w = ((1/((np.pi*m)**2))*(u_x)**2 + (1/((np.pi*n)**2))*(u_y)**2 + self.c) ** (1/2)
        
        return w

    def net_sample(self, t, x, y):
        inputs = torch.cat([t, x, y], dim=1)
        xy_new = self.dnn(inputs)
        x_new = xy_new[:, [0]]
        y_new = xy_new[:, [1]]

        gt0 = t - self.lb[0]   # ramp: =0 at t=0, grows with t
        gx0 = x - self.lb[1]
        gx1 = x - self.ub[1]
        gy0 = y - self.lb[2]
        gy1 = y - self.ub[2]

        x_new = gx0 * gx1 * x_new + x
        y_new = gy0 * gy1 * y_new + y
       # x_new = gt0 * gx0 * gx1 * x_new + x
       # y_new = gt0 * gy0 * gy1 * y_new + y
        return x_new, y_new

    def net_f(self, t, x, y):
        x_new, y_new = self.net_sample(t, x, y)
        x_new_t = torch.autograd.grad(
            x_new, t, grad_outputs=torch.ones_like(x_new),
            retain_graph=True, create_graph=True)[0]
        y_new_t = torch.autograd.grad(
            y_new, t, grad_outputs=torch.ones_like(y_new),
            retain_graph=True, create_graph=True)[0]
        x_new_x = torch.autograd.grad(
            x_new, x, grad_outputs=torch.ones_like(x_new),
            retain_graph=True, create_graph=True)[0]
        x_new_y = torch.autograd.grad(
            x_new, y, grad_outputs=torch.ones_like(x_new),
            retain_graph=True, create_graph=True)[0]
        y_new_x = torch.autograd.grad(
            y_new, x, grad_outputs=torch.ones_like(y_new),
            retain_graph=True, create_graph=True)[0]
        y_new_y = torch.autograd.grad(
            y_new, y, grad_outputs=torch.ones_like(y_new),
            retain_graph=True, create_graph=True)[0]
        G = self.monitor(x, y)
        G_x = torch.autograd.grad(
            G, x, grad_outputs=torch.ones_like(G),
            retain_graph=True, create_graph=True)[0]
        G_y = torch.autograd.grad(
            G, y, grad_outputs=torch.ones_like(G),
            retain_graph=True, create_graph=True)[0]
        J = x_new_x * y_new_y - x_new_y * y_new_x
        A = (x_new_y**2 + y_new_y**2) / (G * J)
        B = (x_new_x * x_new_y + y_new_x * y_new_y) / (G * J)
        C_mat = (x_new_x**2 + y_new_x**2) / (G * J)


        A_x = torch.autograd.grad(
            A, x, grad_outputs=torch.ones_like(A),
            retain_graph=True, create_graph=True)[0]
        B_x = torch.autograd.grad(
            B, x, grad_outputs=torch.ones_like(B),
            retain_graph=True, create_graph=True)[0]
        B_y = torch.autograd.grad(
            B, y, grad_outputs=torch.ones_like(B),
            retain_graph=True, create_graph=True)[0]
        C_y = torch.autograd.grad(
            C_mat, y, grad_outputs=torch.ones_like(C_mat),
            retain_graph=True, create_graph=True)[0]
        
        f_x = (x_new_t * self.nu 
               + x_new_x * (A_x - B_y)
               + x_new_y * (C_y - B_x))
        f_y = (y_new_t * self.nu 
               + y_new_x * (A_x - B_y)
               + y_new_y * (C_y - B_x))
        return f_x, f_y

    def loss_func(self):
        f_x, f_y = self.net_f(self.t_f, self.x_f, self.y_f)
        return torch.mean(f_x**2) + torch.mean(f_y**2)

    def optimize_one_epoch(self):
        if self.start_time is None:
            self.start_time = time.time()
        self.optimizer.zero_grad()
        self.loss = self.loss_func()
        self.loss.backward()
        self.iter += 1
        if self.iter % 100 == 0:
            loss = self.detach(self.loss)
            print(f"{self.optimizer_name} Iter {self.iter}, Loss: {loss:.6f}")
            elapsed = time.time() - self.start_time
            print(f'Time: {elapsed:.4f}s')
            self.start_time = time.time()
        return self.loss

    def train_Adam(self, optimizer, nIter):
        self.optimizer = optimizer
        self.optimizer_name = 'MMPDE_initial_new_Adam'
        for it in range(nIter):
            self.optimize_one_epoch()
            self.optimizer.step()

    def train_LBFGS(self, optimizer):
        self.optimizer = optimizer
        self.optimizer_name = 'MMPDE_initial_new_LBFGS'
        def closure():
            return self.optimize_one_epoch()
        self.optimizer.step(closure)

    def train(self):
        print("开始 sampling_MMPDE_initial_new 训练...")
        self.train_Adam(self.optimizer_Adam, self.AdamIter)
        print("sampling_MMPDE_initial_new Adam Done!")
        self.train_LBFGS(self.optimizer_LBFGS)
        print('sampling_MMPDE_initial_new LBFGS Done!')

        # 仅取 t=T 时刻对应的 nx*ny 个空间点（去除重复的时间层）
        mask = (self.t_f.squeeze() >= self.ub[0] - 1e-8)
        x_T = self.x_f[mask].detach().requires_grad_(True)
        y_T = self.y_f[mask].detach().requires_grad_(True)
        t_T = (self.ub[0] * torch.ones_like(x_T)).detach().requires_grad_(True)
        x_new, y_new = self.net_sample(t_T, x_T, y_T)
        new_sample = torch.cat([x_new, y_new], dim=1)
        print(f"[MMPDE train] new_sample shape = {new_sample.shape}  "
              f"(仅 t=T={self.ub[0].item():.4f} 时刻的 {mask.sum().item()} 个空间点)")
        return new_sample


class sampling_MMPDE_2D():
    def __init__(self, X_f, u_fun, layers, lb, ub, nu, AdamIter, LBFGSIter):
        self.lb = torch.tensor(lb).float().to(device)
        self.ub = torch.tensor(ub).float().to(device)

        self.t_f = torch.tensor(X_f[:, 0:1], requires_grad=True).float().to(device)
        self.x_f = torch.tensor(X_f[:, 1:2], requires_grad=True).float().to(device)
        self.y_f = torch.tensor(X_f[:, 2:3], requires_grad=True).float().to(device)
        self.fun = u_fun

        self.layers = layers
        self.nu = nu

        self.dnn = DNN(layers).to(device)

        self.optimizer_Adam = torch.optim.Adam(self.dnn.parameters(),
                                              lr=1e-3,
                                              betas=(0.9, 0.999),
                                              eps=1e-8)
        
        self.AdamIter = AdamIter

        self.optimizer_LBFGS = torch.optim.LBFGS(
            self.dnn.parameters(),
            lr=0.5,
            max_iter=LBFGSIter,
        )

        self.optimizer = None
        self.loss = None
        self.iter = 0
        self.start_time = None

    def detach(self, data):
        return data.detach().cpu().numpy()

    def monitor(self, t, x, y):

        t.requires_grad_(True)
        x.requires_grad_(True) 
        y.requires_grad_(True)

        inputs = torch.cat([t, x, y], dim=1)
        uv = self.fun(inputs)
        u = uv[:, [0]]  
        v = uv[:, [1]]  

        u_x = torch.autograd.grad(
            u, x,
            grad_outputs=torch.ones_like(u),
            retain_graph=True,
            create_graph=True,
            allow_unused=True
        )[0]
        
        u_y = torch.autograd.grad(
            u, y,
            grad_outputs=torch.ones_like(u),
            retain_graph=True,
            create_graph=True,
            allow_unused=True
        )[0]

        v_t = torch.autograd.grad(
            v, t,
            grad_outputs=torch.ones_like(v),
            retain_graph=True,
            create_graph=True,
            allow_unused=True
        )[0]

        if u_x is None:
            u_x = torch.zeros_like(u)
        if u_y is None:
            u_y = torch.zeros_like(u)
        if v_t is None:
            v_t = torch.zeros_like(v)

        w = ((1/((np.pi*m)**2))*(u_x)**2 + (1/((np.pi*n)**2))*(u_y)**2 + 0.01*(v_t)**2) ** (1/2)
        #w = 0.5 *  (c**2 *((u_x)**2 + (u_y)**2)  +  v**2 +  2) 
        #print(f"监控函数 - w: {w.mean().item():.6e}")
        
        return w

    def net_sample(self, t, x, y):
        inputs = torch.cat([t, x, y], dim=1)
        xy_new = self.dnn(inputs)

        x_new = xy_new[:, [0]]
        y_new = xy_new[:, [1]]
        
        gt0 = t - self.lb[0] 

        gx0 = x - self.lb[1] 
        gx1 = x - self.ub[1] 
        gy0 = y - self.lb[2] 
        gy1 = y - self.ub[2]

        #x_new =  gx0 * gx1 * x_new + x
        #y_new =  gy0 * gy1 * y_new + y

        x_new = gt0 * gx0 * gx1 * x_new + x
        y_new = gt0 * gy0 * gy1 * y_new + y
        
        return x_new, y_new

    def net_f(self, t, x, y):
        x_new, y_new = self.net_sample(t, x, y)

        x_new_t = torch.autograd.grad(
            x_new, t,
            grad_outputs=torch.ones_like(x_new),
            retain_graph=True,
            create_graph=True
        )[0]

        y_new_t = torch.autograd.grad(
            y_new, t,
            grad_outputs=torch.ones_like(y_new),
            retain_graph=True,
            create_graph=True
        )[0]

        x_new_x = torch.autograd.grad(
            x_new, x,
            grad_outputs=torch.ones_like(x_new),
            retain_graph=True,
            create_graph=True
        )[0]

        x_new_y = torch.autograd.grad(
            x_new, y,
            grad_outputs=torch.ones_like(x_new),
            retain_graph=True,
            create_graph=True
        )[0]

        y_new_x = torch.autograd.grad(
            y_new, x,
            grad_outputs=torch.ones_like(y_new),
            retain_graph=True,
            create_graph=True
        )[0]

        y_new_y = torch.autograd.grad(
            y_new, y,
            grad_outputs=torch.ones_like(y_new),
            retain_graph=True,
            create_graph=True
        )[0]

        x_new_xx = torch.autograd.grad(
            x_new_x, x,
            grad_outputs=torch.ones_like(x_new_x),
            retain_graph=True,
            create_graph=True
        )[0]

        x_new_yy = torch.autograd.grad(
            x_new_y, y,
            grad_outputs=torch.ones_like(x_new_y),
            retain_graph=True,
            create_graph=True
        )[0]

        y_new_xx = torch.autograd.grad(
            y_new_x, x,
            grad_outputs=torch.ones_like(y_new_x),
            retain_graph=True,
            create_graph=True
        )[0]

        y_new_yy = torch.autograd.grad(
            y_new_y, y,
            grad_outputs=torch.ones_like(y_new_y),
            retain_graph=True,
            create_graph=True
        )[0]

        G = self.monitor(t, x, y)

        G_t = torch.autograd.grad(
            G, t,
            grad_outputs=torch.ones_like(G),
            retain_graph=True,
            create_graph=True
        )[0]

        G_x = torch.autograd.grad(
            G, x,
            grad_outputs=torch.ones_like(G),
            retain_graph=True,
            create_graph=True
        )[0]

        G_y = torch.autograd.grad(
            G, y,
            grad_outputs=torch.ones_like(G),
            retain_graph=True,
            create_graph=True
        )[0]
        
        J = x_new_x * y_new_y - x_new_y * y_new_x  # Jocabi
        A = (x_new_y**2 + y_new_y**2) / (G*J)
        B = (x_new_x * x_new_y + y_new_x * y_new_y) / (G*J)
        C = (x_new_x**2 + y_new_x**2) / (G*J)
        
        A_x = torch.autograd.grad(
            A, x,
            grad_outputs=torch.ones_like(A),
            retain_graph=True,
            create_graph=True
        )[0]
        
        
        B_x = torch.autograd.grad(
            B, x,
            grad_outputs=torch.ones_like(B),
            retain_graph=True,
            create_graph=True
        )[0]
        
        B_y = torch.autograd.grad(
            B, x,
            grad_outputs=torch.ones_like(B),
            retain_graph=True,
            create_graph=True
        )[0]
        
        C_y = torch.autograd.grad(
            C, y,
            grad_outputs=torch.ones_like(C),
            retain_graph=True,
            create_graph=True
        )[0]
        
        
        # For x coordinate:
        #E_x = G_x * x_new_x + G_y * x_new_y + G * (x_new_xx + x_new_yy)
        #f_x = x_new_t * self.nu * (G**2) * (x_new_x**2 + x_new_y**2) + E_x
        f_x = (x_new_t * self.nu #* J 
               + x_new_x * (A_x-B_y)
               + x_new_y * (C_y-B_x))
        
        
        # For y coordinate:
        #E_y = G_x * y_new_x + G_y * y_new_y + G * (y_new_xx + y_new_yy)
        #f_y = y_new_t * self.nu * (G**2) * (y_new_x**2 + y_new_y**2) + E_y
        f_y = (x_new_t * self.nu #* J 
               + y_new_x * (A_x-B_y)
               + y_new_y * (C_y-B_x))
        
        return f_x, f_y

    def loss_func(self):
        f_x, f_y = self.net_f(self.t_f, self.x_f, self.y_f)
        loss_f = torch.mean(f_x ** 2) + torch.mean(f_y ** 2)
        
        return loss_f

    def optimize_one_epoch(self):
        if self.start_time is None:
            self.start_time = time.time()

        self.optimizer.zero_grad()
        self.loss = self.loss_func()
        self.loss.backward()
        self.iter = self.iter + 1

        if self.iter % 100 == 0:
            loss = self.detach(self.loss)
            print(f"{self.optimizer_name} Iter {self.iter}, Loss: {loss:.6f}")
            elapsed = time.time() - self.start_time
            print(f'Time: {elapsed:.4f}s')
            self.start_time = time.time()

        return self.loss

    def train_Adam(self, optimizer, nIter):
        self.optimizer = optimizer
        self.optimizer_name = '2D_EMMPDE_Adam'
        
        for it in range(nIter):
            self.optimize_one_epoch()
            self.optimizer.step()

    def train_LBFGS(self, optimizer):
        self.optimizer = optimizer
        self.optimizer_name = '2D_EMMPDE_LBFGS'

        def closure():
            loss = self.optimize_one_epoch()
            return loss

        self.optimizer.step(closure)

    def train(self):
        print("开始2D MMPDE训练...")
        self.train_Adam(self.optimizer_Adam, self.AdamIter)
        print("2D MMPDE_Adam 完成!")
        self.train_LBFGS(self.optimizer_LBFGS)
        print('2D MMPDE_LBGFS 完成!')

        # 返回新的采样点
        x_new, y_new = self.net_sample(self.t_f, self.x_f, self.y_f)
        new_sample = torch.cat([self.t_f, x_new, y_new], dim=1)
        return new_sample

In [ ]:

################################################################################
# Neural Network Definition
################################################################################
class FourierFeatureMap(nn.Module):
    def __init__(self,  n_input, n_output, sigma=1.0, mode='none'):
        super().__init__()
        allowed_modes = {'none', 'spatial', 'spacetime'}
        if mode not in allowed_modes:
            raise ValueError(f"fourier mode must be one of {allowed_modes}, got {mode!r}")
        self.mode = mode
        #self.include_raw = include_raw
        #freqs = torch.arange(1, num_frequencies + 1, dtype=torch.float32)
        #self.register_buffer('freqs', freqs)

        self.B = torch.nn.Parameter(torch.normal(0, sigma, size=(n_input, n_output)), requires_grad=False).to(device)

   
    def forward(self, x):
        if self.mode == 'none':
            return x

        #coords = x[:, 1:3] if self.mode == 'spatial' else x
        #angles = 2.0 * np.pi * coords.unsqueeze(-1) * self.freqs
        fourier_features = torch.cat([torch.sin(torch.matmul(x, self.B)), torch.cos(torch.matmul(x, self.B))], dim=-1)
       
        #if self.include_raw:
        #   return torch.cat([x, fourier_features], dim=1)
        return fourier_features


class PINN_FF(nn.Module):
    def __init__(self, layers, activation=nn.Tanh(), n_fourier_features=64, fourier_mode='none'):
        super(PINN_FF, self).__init__()
        
        self.feature_map = FourierFeatureMap(
            mode=fourier_mode,
            n_input=layers[0],
            n_output=n_fourier_features//2,
        )
        mapped_input_dim = n_fourier_features
        effective_layers = [mapped_input_dim] + layers[1:]

        self.linears = nn.ModuleList()
        for i in range(len(effective_layers) - 1):
            self.linears.append(nn.Linear(effective_layers[i], effective_layers[i + 1]))

        self.activation = activation

        # Initialize weights (Xavier initialization)
        for m_layer in self.linears:
            nn.init.xavier_normal_(m_layer.weight)
            nn.init.zeros_(m_layer.bias)

    def forward(self, x):
        x = self.feature_map(x)
        for i in range(len(self.linears) - 1):
            x = self.activation(self.linears[i](x))
        # output: [u, v] where v = u_t
        x = self.linears[-1](x)
        return x


In [ ]:
class PINN(nn.Module):
    def __init__(self, layers, activation=nn.Tanh()):
        super(PINN, self).__init__()
        
        self.linears = nn.ModuleList()
        for i in range(len(layers) - 1):
            self.linears.append(nn.Linear(layers[i], layers[i+1]))
        
        self.activation = activation

        for m in self.linears:
            nn.init.xavier_normal_(m.weight)
            nn.init.zeros_(m.bias)
    
    def forward(self, x):
        for i in range(len(self.linears) - 1):
            x = self.activation(self.linears[i](x))
            
        x = self.linears[-1](x)
        return x

In [ ]:
def f1(x, y, m_val=None, n_val=None):
    m_eff = m if m_val is None else m_val
    n_eff = n if n_val is None else n_val
    return np.sin(m_eff * np.pi * x) * np.sin(n_eff * np.pi * y)


def f1_tensor(x, y, m_val=None, n_val=None):
    m_eff = m if m_val is None else m_val
    n_eff = n if n_val is None else n_val
    return torch.sin(m_eff * np.pi * x) * torch.sin(n_eff * np.pi * y)


def f2(x, y):
    return np.zeros_like(x)

def boundary_condition_u(t, x, y):
    return np.zeros_like(t)

def boundary_condition_v(t, x, y):
    return np.zeros_like(t)

In [ ]:
def exact_solution_u(x, y, t, c=1.0, m_val=None, n_val=None):
    """
    Exact analytical solution for u:
    u(x,y,t) = sin(m*pi*x) * sin(n*pi*y) * cos(c*pi*sqrt(m^2+n^2)*t)
    """
    m_eff = m if m_val is None else m_val
    n_eff = n if n_val is None else n_val
    omega = np.pi * c * np.sqrt(m_eff**2 + n_eff**2)
    return np.sin(m_eff * np.pi * x) * np.sin(n_eff * np.pi * y) * np.cos(omega * t)

def exact_solution_v(x, y, t, c=1.0, m_val=None, n_val=None):
    """
    Exact analytical solution for v = du/dt:
    v(x,y,t) = -omega * sin(m*pi*x) * sin(n*pi*y) * sin(omega*t)
    """
    m_eff = m if m_val is None else m_val
    n_eff = n if n_val is None else n_val
    omega = np.pi * c * np.sqrt(m_eff**2 + n_eff**2)
    return -omega * np.sin(m_eff * np.pi * x) * np.sin(n_eff * np.pi * y) * np.sin(omega * t)

def exact_solution_uv(x, y, t, c=1.0, m_val=None, n_val=None):
    u = exact_solution_u(x, y, t, c, m_val, n_val)
    v = exact_solution_v(x, y, t, c, m_val, n_val)
    return np.column_stack((u.flatten(), v.flatten()))


In [ ]:
def wave_pde_loss(model, t, x, y):
    t.requires_grad_(True)
    x.requires_grad_(True)
    y.requires_grad_(True)

    inputs = torch.cat((t, x, y), dim=1)
    uv = model(inputs)
    u = uv[:, [0]] 
    v = uv[:, [1]]  

    u_grads = torch.autograd.grad(u, inputs, 
                                 grad_outputs=torch.ones_like(u), 
                                 create_graph=True)[0]
    u_t = u_grads[:, [0]]
    u_x = u_grads[:, [1]]
    u_y = u_grads[:, [2]]

    v_grads = torch.autograd.grad(v, inputs,
                                 grad_outputs=torch.ones_like(v),
                                 create_graph=True)[0]
    v_t = v_grads[:, [0]]

    u_xx = torch.autograd.grad(u_x, x,
                              grad_outputs=torch.ones_like(u_x),
                              create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, y,
                              grad_outputs=torch.ones_like(u_y),
                              create_graph=True)[0]

    f_u = u_t - v

    f_v = v_t - (c**2) * (u_xx + u_yy)

    return torch.mean(f_u**2) + torch.mean(f_v**2)


def wave_pde_f(model, t, x, y):
    t.requires_grad_(True)
    x.requires_grad_(True)
    y.requires_grad_(True)

    inputs = torch.cat((t, x, y), dim=1)
    uv = model(inputs)
    u = uv[:, [0]] 
    v = uv[:, [1]]  

    u_grads = torch.autograd.grad(u, inputs, 
                                 grad_outputs=torch.ones_like(u), 
                                 create_graph=True)[0]
    u_t = u_grads[:, [0]]
    u_x = u_grads[:, [1]]
    u_y = u_grads[:, [2]]

    v_grads = torch.autograd.grad(v, inputs,
                                 grad_outputs=torch.ones_like(v),
                                 create_graph=True)[0]
    v_t = v_grads[:, [0]]

    u_xx = torch.autograd.grad(u_x, x,
                              grad_outputs=torch.ones_like(u_x),
                              create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, y,
                              grad_outputs=torch.ones_like(u_y),
                              create_graph=True)[0]

    #f_u = u_t - v

    f_v = v_t - (c**2) * (u_xx + u_yy)

    return f_v


def initial_condition_loss(model, t, x, y, uv_true):
    inputs = torch.cat((t, x, y), dim=1)
    uv_pred = model(inputs)

    u = uv_pred[:, [0]] 
    v = uv_pred[:, [1]]
    
    u_true = uv_true[:, [0]] 
    v_true = uv_true[:, [1]]
    
    ic_u_loss = torch.mean((u - u_true)**2)
    ic_v_loss = torch.mean((v - v_true)**2)
    
    #ic_loss = torch.mean((uv_pred - uv_true)**2)
    
    return ic_u_loss, ic_v_loss


def boundary_condition_loss(model, boundary_pts):
    bc_loss = 0
    
    # Process each boundary
    for boundary_name, (t, x, y, uv_true) in boundary_pts.items():
        inputs = torch.cat((t, x, y), dim=1)
        uv_pred = model(inputs)
        u_true = uv_true[:, [0]]
        u_pred = uv_pred[:, [0]]

        bc_loss += torch.mean((u_pred - u_true)**2)
    
    return bc_loss


def loss_function(model, collocation_pts, initial_pts, boundary_pts, lam_u=1.0, lam_ut=1.0, lam_r=1.0):

    t_coll, x_coll, y_coll = collocation_pts
    t_init, x_init, y_init, uv_init = initial_pts

    pde_loss = wave_pde_loss(model, t_coll, x_coll, y_coll)

    ic_u_loss, ic_v_loss = initial_condition_loss(model, t_init, x_init, y_init, uv_init)

    bc_loss = boundary_condition_loss(model, boundary_pts)

    bc_loss = ic_u_loss + bc_loss

    total_loss =  lam_r*pde_loss + lam_ut*ic_v_loss +  lam_u*bc_loss

    return total_loss, pde_loss, ic_v_loss, bc_loss

In [ ]:
def plot_initial_samples(x_init_new, y_init_new, Lx=Lx, Ly=Ly,
                         color='red', s=8, alpha=0.6, figsize=(5, 5)):
    if isinstance(x_init_new, torch.Tensor):
        x_plot = x_init_new.detach().cpu().numpy().flatten()
    else:
        x_plot = np.asarray(x_init_new).flatten()
    if isinstance(y_init_new, torch.Tensor):
        y_plot = y_init_new.detach().cpu().numpy().flatten()
    else:
        y_plot = np.asarray(y_init_new).flatten()

    plt.figure(figsize=figsize)
    plt.scatter(x_plot, y_plot, s=s, c=color, alpha=alpha)
    plt.xlim(0, Lx); plt.ylim(0, Ly)
    plt.gca().set_aspect('equal')
    plt.xlabel('$x_1$', fontsize=20)
    plt.ylabel('$x_2$', fontsize=20)
    plt.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.show()


In [ ]:
################################################################################
#迭代训练策略：PINN + MMPDE 交替优化
################################################################################
def create_2d_simulation_function_from_model(model):
    """基于当前PINN模型创建模拟函数"""
    def simulation_function(txy):
        model.eval()  # 设置为评估模式
        # 注意：不使用torch.no_grad()，因为MMPDE需要计算梯度
        txy = txy.to(device)
        
        # 确保输入张量需要梯度
        if not txy.requires_grad:
            txy.requires_grad_(True)
            
        return model(txy)
    return simulation_function

def generate_uniform_collocation_points():    
    # not include the initial time slice t=0, only (0,T] for better MMPDE training
    #n_per_dim = int(np.cbrt(n_points))  

    t_uniform = torch.linspace(T/(nt), T, nt, device=device)
    x_uniform = torch.linspace(0, Lx, nx, device=device)  
    y_uniform = torch.linspace(0, Ly, ny, device=device)

    T_grid, X_grid, Y_grid = torch.meshgrid(t_uniform, x_uniform, y_uniform, indexing='ij')

    t_coll = T_grid.reshape(-1, 1)
    x_coll = X_grid.reshape(-1, 1)
    y_coll = Y_grid.reshape(-1, 1)

    t_coll.requires_grad_(True)
    x_coll.requires_grad_(True)
    y_coll.requires_grad_(True)
    
    print(f"生成等距配点网格: {nt}×{nx}×{ny} = {t_coll.shape[0]}个点")
    
    return (t_coll, x_coll, y_coll)

def generate_X_f_for_mmpde():
    """
    生成 X_f numpy [N, 3]，列依次为 (t, x, y)，
    覆盖时空域 [0,T] × [0,Lx] × [0,Ly]，用于 sampling_MMPDE_initial_new。
    """
    t_uniform = np.linspace(0, T, nt)
    x_uniform = np.linspace(0, Lx, nx)
    y_uniform = np.linspace(0, Ly, ny)

    T_grid, X_grid, Y_grid = np.meshgrid(t_uniform, x_uniform, y_uniform, indexing='ij')

    X_f = np.column_stack([T_grid.flatten(), X_grid.flatten(), Y_grid.flatten()])
    return X_f   # shape: (nt * nx * ny, 3)

def generate_initial_training_data():
    collocation_pts = generate_uniform_collocation_points()

    '''
    x_init_grid, y_init_grid = np.meshgrid(np.linspace(0, Lx, nx), 
                                         np.linspace(0, Ly, ny))
    
    x_init = x_init_grid.flatten().reshape(-1, 1)
    y_init = y_init_grid.flatten().reshape(-1, 1)

    t_init = np.zeros_like(x_init)
    '''
    

    # from MATLAB 预先生成的初始采样点 (x,y) 网格坐标
    data = scipy.io.loadmat(f'mesh_{nx}_{ny}_{m}_{n}.mat')
    expected_shape = (nx, ny)
    if data['x_tT'].shape != expected_shape or data['y_tT'].shape != expected_shape:
        raise ValueError(f"x_tT and y_tT must both have shape {expected_shape}")
    x_init = data['x_tT'].flatten().reshape(-1, 1)  # shape: (N1*N2,)
    y_init = data['y_tT'].flatten().reshape(-1, 1)
    if x_init.shape[0] != nx * ny:
        raise ValueError(f'the fixed initial mesh must contain {nx * ny} points')



    '''

    mmpde_init_layer = [2,50,50,50,50,2]

    new_init_sampling = sampling_MMPDE_initial(x_init, y_init, u_fun=f1_tensor, 
                                      layers=mmpde_init_layer,
                                       lb=[0.0, 0.0], ub=[Lx, Ly], c=1.0,
                                        dt=0.01, nu=50, AdamIter=100, LBFGSIter=0)
    '''
    
    '''
    # use trained PINN to generate the monitor function for MMPDE, and train MMPDE to get new initial sampling points at t=T
    X_f = generate_X_f_for_mmpde()
    
    new_init_sampling = sampling_MMPDE_initial_new(
                              X_f=X_f,             # numpy [N,3]
                              u_fun=f1_tensor,
                              layers=[3,50,50,50,50,2],
                              lb=[0.0, 0.0, 0.0], ub=[T, Lx, Ly],
                              c=0.01, nu=0.001,
                              AdamIter=2000, LBFGSIter=500)
    
    # 训练MMPDE并获取新初值的采样点 (返回 tensor)
    new_init_samples = new_init_sampling.train()
    print(f"[主程序] new_init_samples shape: {new_init_samples.shape}  "
          f"← 应为 [nx*ny, 2] = [{nx*ny}, 2]，仅 t=T 时刻的 (x_new, y_new)")

    # 提取新采样点 (保持 tensor)
    x_init_new = new_init_samples[:, 0:1].clone().detach().to(device).requires_grad_(True)
    y_init_new = new_init_samples[:, 1:2].clone().detach().to(device).requires_grad_(True)
    

    
    # 可视化新采样点

    # 用于 numpy 版 f1 / f2 计算初始条件值
    x_init_new_np = x_init_new.detach().cpu().numpy()
    y_init_new_np = y_init_new.detach().cpu().numpy()
    ##
    '''

    
    # not learning intial sampling, just use the original grid points as initial sampling for PINN training
    x_init_new_np = x_init
    y_init_new_np = y_init
    x_init_new = torch.tensor(x_init_new_np, dtype=torch.float32, device=device, requires_grad=True)
    y_init_new = torch.tensor(y_init_new_np, dtype=torch.float32, device=device, requires_grad=True)
    ######
    


    u_init = f1(x_init_new_np, y_init_new_np, m, n)
    v_init = f2(x_init_new_np, y_init_new_np)
    uv_init = np.column_stack((u_init.flatten(), v_init.flatten()))

    # t_init 同样要匹配新采样点数量
    t_init = np.zeros_like(x_init_new_np)

    t_init = torch.tensor(t_init, dtype=torch.float32, device=device, requires_grad=True)
    # 直接用新采样点张量作为 x_init / y_init
    x_init = x_init_new
    y_init = y_init_new
    uv_init = torch.tensor(uv_init, dtype=torch.float32, device=device)

    t_bound_bottom = torch.rand(n_boundary, 1, device=device) * T
    x_bound_bottom = torch.rand(n_boundary, 1, device=device) * Lx
    y_bound_bottom = torch.zeros(n_boundary, 1, device=device)
    
    t_bound_top = torch.rand(n_boundary, 1, device=device) * T
    x_bound_top = torch.rand(n_boundary, 1, device=device) * Lx
    y_bound_top = Ly * torch.ones(n_boundary, 1, device=device)
    
    t_bound_left = torch.rand(n_boundary, 1, device=device) * T
    x_bound_left = torch.zeros(n_boundary, 1, device=device)
    y_bound_left = torch.rand(n_boundary, 1, device=device) * Ly
    
    t_bound_right = torch.rand(n_boundary, 1, device=device) * T
    x_bound_right = Lx * torch.ones(n_boundary, 1, device=device)
    y_bound_right = torch.rand(n_boundary, 1, device=device) * Ly

    uv_bound_bottom = torch.zeros(n_boundary, 2, device=device)
    uv_bound_top = torch.zeros(n_boundary, 2, device=device)
    uv_bound_left = torch.zeros(n_boundary, 2, device=device)
    uv_bound_right = torch.zeros(n_boundary, 2, device=device)

    for tensor in [t_bound_bottom, x_bound_bottom, y_bound_bottom,
                  t_bound_top, x_bound_top, y_bound_top,
                  t_bound_left, x_bound_left, y_bound_left,
                  t_bound_right, x_bound_right, y_bound_right]:
        tensor.requires_grad_(True)
    
    initial_pts = (t_init, x_init, y_init, uv_init)
    boundary_pts = {
        'bottom': (t_bound_bottom, x_bound_bottom, y_bound_bottom, uv_bound_bottom),
        'top': (t_bound_top, x_bound_top, y_bound_top, uv_bound_top),
        'left': (t_bound_left, x_bound_left, y_bound_left, uv_bound_left),
        'right': (t_bound_right, x_bound_right, y_bound_right, uv_bound_right)
    }

    
    return collocation_pts, initial_pts, boundary_pts


In [ ]:
# ── NTK utilities ──────────────────────────────────────────────────────
def compute_jacobian(model, f):
        """Compute Jacobian J[i, :] = d f[i,0] / d theta.

        Args:
            model: PyTorch model (nn.Module).
            f: Tensor [N, 1] — network output (built with create_graph=True
               or from a graph that allows grad computation).
        Returns:
            J: Tensor [N, P] on CPU (P = total number of parameters).
        """
        params = list(model.parameters())
        N = f.shape[0]
        rows = []
        for i in range(N):
            grads = torch.autograd.grad(
                f[i, 0], params,
                retain_graph=True,
                allow_unused=True
            )
            row = torch.cat([
                (torch.zeros_like(p) if g is None else g).flatten()
                for g, p in zip(grads, params)
            ])
            rows.append(row.detach())
        return torch.stack(rows)   # [N, P]


def compute_ntk(J1, J2):
        """NTK = J1 @ J2^T  ->  numpy array."""
        return (J1 @ J2.T).cpu().numpy()

In [ ]:
################################################################################
# 修改后的训练函数
################################################################################
def iterative_training_with_adaptive_sampling_updated(
    initial_epochs,      # 初始训练轮数
    adaptive_epochs,     # 每次自适应采样后的训练轮数  
    num_iterations,         # 迭代次数
    mmpde_training_epochs, # MMPDE训练轮数
    lam_u  = 1.0,
    lam_ut = 1.0,
    lam_r  = 1.0
):   
    
    lam_u_log =[]
    lam_ut_log = []
    lam_r_log = []
    K_u_log = []
    K_ut_log = []
    K_r_log = []
    loss_history = []
    
    print("=" * 80)
    print(f"开始迭代训练：PINN + MMPDE 自适应采样 (时间域: [0, {T:.4f}])")
    print("=" * 80)
    
    # ==================== 第一阶段：初始训练 ====================
    print(f"\n【第一阶段】初始训练 ({initial_epochs} epochs)")
    
    # 创建初始模型
    layers = [3] + [neurons]*hidden_layers + [2]
    model = PINN_FF(
    layers,
    activation=nn.Tanh(),
    n_fourier_features=64,
    fourier_mode=fourier_mode).to(device)


    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    # 初始训练数据
    collocation_pts, initial_pts, boundary_pts = generate_initial_training_data()
    

    # 存储采样点历史用于可视化
    sampling_points_history = []
    
    # 原配置点不包括初始值点，现在将初始条件t=0的点，合并到配点中进行训练
    t_coll, x_coll, y_coll = collocation_pts
    t_init, x_init, y_init, _ = initial_pts

    t_coll = torch.cat([t_init.detach(), t_coll], dim=0).clone().detach().requires_grad_(True)
    x_coll = torch.cat([x_init.detach(), x_coll], dim=0).clone().detach().requires_grad_(True)
    y_coll = torch.cat([y_init.detach(), y_coll], dim=0).clone().detach().requires_grad_(True)
    collocation_pts = (t_coll, x_coll, y_coll)

    print(f"合并后 collocation 点数: {t_coll.shape[0]}") 

    initial_samples = torch.cat([t_coll, x_coll, y_coll], dim=1).detach().cpu().numpy()
    sampling_points_history.append(("Initial Sampling", initial_samples))
    
    current_sampling_points = collocation_pts
    
    # 初始训练
    print("开始初始训练...")
    
    model.train()

    for epoch in range(initial_epochs):
        optimizer.zero_grad()
        loss, pde_loss, ic_loss, bc_loss = loss_function(model, collocation_pts, initial_pts, boundary_pts,
                                                         lam_u=lam_u, lam_ut=lam_ut, lam_r=lam_r)
        
        loss.backward()
        optimizer.step()
        loss_history.append((loss.item(), pde_loss.item(), ic_loss.item(), bc_loss.item()))
        
        if (epoch + 1) % 500 == 0:
           print(f"Epoch [{epoch+1}/{initial_epochs}], "
                 f"Loss: {loss.item():.6e}, "
                 f"PDE Loss: {pde_loss.item():.6e}, "
                 f"IC Loss: {ic_loss.item():.6e}, "
                 f"BC Loss: {bc_loss.item():.6e}")
           
           print(f"[NTK] Epoch {epoch+1}: lam_r={lam_r:.4f},lam_ut={lam_ut:.4f}, lam_u={lam_u:.4f}")
        
        if log_NTK and epoch % 100 == 0:
                print('Computing NTK...')
                model.eval()
                D = kernel_size

                # Fresh samples for NTK

                t_ic, x_ic, y_ic, _ = initial_pts
                txy_ic = torch.cat([t_ic.detach(), x_ic.detach(), y_ic.detach()], dim=1)
      
                 # 提取所有 boundary_pts 的 (t, x, y) 并合并
                txy_bc_list = []
                for _, (t_b, x_b, y_b, _) in boundary_pts.items():
                    txy_bc_list.append(torch.cat([t_b.detach(), x_b.detach(), y_b.detach()], dim=1))
                txy_bc = torch.cat(txy_bc_list, dim=0)

                # 合并 initial + boundary
                X_bc_ntk_all = torch.cat([txy_ic, txy_bc], dim=0)  # [N_ic + N_bc_total, 3]

                # 随机取 D 点
                N_bc_total = X_bc_ntk_all.shape[0]
                idx_bc = torch.randperm(N_bc_total)[:D]
                X_bc_ntk = X_bc_ntk_all[idx_bc]  # [D, 3]

                uv_ntk_pred = model(X_bc_ntk)

                u_ntk_pred = uv_ntk_pred[:, [0]] 

                # X_ut_ntk: 从 initial_pts 中随机抽取 D 个点，用于 u_t 损失的 NTK
                t_init_d, x_init_d, y_init_d, _ = initial_pts
                N_init = t_init_d.shape[0]
                idx_ut = torch.randperm(N_init)[:D]
                t_ut = t_init_d[idx_ut].clone().detach().requires_grad_(True)
                x_ut = x_init_d[idx_ut].clone().detach().requires_grad_(True)
                y_ut = y_init_d[idx_ut].clone().detach().requires_grad_(True)

                # X_r_ntk: 从 collocation_pts 中随机抽取 D 个点，用于 PDE residual 的 NTK
                t_c, x_c, y_c = collocation_pts
                N_coll = t_c.shape[0]
                idx_r = torch.randperm(N_coll)[:D]
                t_r = t_c[idx_r].clone().detach().requires_grad_(True)
                x_r = x_c[idx_r].clone().detach().requires_grad_(True)
                y_r = y_c[idx_r].clone().detach().requires_grad_(True)

                # 计算 u_t 预测（初始条件处）
                inputs_ut = torch.cat([t_ut, x_ut, y_ut], dim=1)
                uv_ut = model(inputs_ut)
                u_t_pred = uv_ut[:, [1]]  # v 代表 u_t 的预测值

                 # 计算 PDE residual 预测
                r_pred_ntk = wave_pde_f(model, t_r, x_r, y_r)  # 返回残差向量

                 # 计算 Jacobian 和 NTK
                J_u  = compute_jacobian(model, u_ntk_pred)
                J_ut = compute_jacobian(model, u_t_pred)
                J_r  = compute_jacobian(model, r_pred_ntk)

                K_u  = compute_ntk(J_u,  J_u)
                K_ut = compute_ntk(J_ut, J_ut)
                K_r  = compute_ntk(J_r,  J_r)


                K_u_log.append(K_u)
                K_ut_log.append(K_ut)
                K_r_log.append(K_r)

                if update_lam:
                    trace_K  = np.trace(K_u) + np.trace(K_ut) + np.trace(K_r)
                    lam_u  =  min(trace_K / np.trace(K_u), 500)
                    #trace_K / np.trace(K_u)
                    lam_ut = min(trace_K / np.trace(K_ut),500)
                    lam_r  = trace_K / np.trace(K_r)
                    lam_u_log.append(lam_u)
                    lam_ut_log.append(lam_ut)
                    lam_r_log.append(lam_r)
                

                model.train()   
            

      
    print(f"初始训练完成 - Loss: {loss.item():.6e}, "
          f"PDE Loss: {pde_loss.item():.6e}, "
          f"IC Loss: {ic_loss.item():.6e}, "
          f"BC Loss: {bc_loss.item():.6e}")

    
    # ==================== 迭代阶段：MMPDE + 继续训练 ====================
    for iteration in range(num_iterations):
        print(f"\n【第{iteration+2}阶段】迭代 {iteration+1}: MMPDE自适应采样 + 继续训练")
        
        # 基于当前模型进行MMPDE采样
        print("基于当前PINN模型进行MMPDE自适应采样...")
        
        # 使用当前模型创建模拟函数
        current_sim_func = create_2d_simulation_function_from_model(model)
        
        # MMPDE参数
        mmpde_layers_2d = [3, 50, 50, 50, 50, 2]
        adam_iter = mmpde_training_epochs
        lbfgs_iter = 0 #  mmpde_training_epochs // 4
        
        # 将 current_sampling_points 转为 numpy [N,3]（首次为 tuple，后续为 numpy array）
        if isinstance(current_sampling_points, tuple):
            t_c, x_c, y_c = current_sampling_points
            X_f_mmpde = torch.cat([t_c.detach(), x_c.detach(), y_c.detach()], dim=1).cpu().numpy()
        else:
            X_f_mmpde = current_sampling_points  # 已经是 numpy array

        # 创建MMPDE采样器
        mmpde_sampler = sampling_MMPDE_2D(
            X_f=X_f_mmpde,
            u_fun=current_sim_func,
            layers=mmpde_layers_2d,
            lb=[0.0, 0.0, 0.0],
            ub=[T, Lx, Ly],  
            nu=0.01,
            AdamIter=adam_iter,
            LBFGSIter=lbfgs_iter
        )
        
        # 训练MMPDE并获取新采样点
        new_samples = mmpde_sampler.train()
        
        # 保存自适应采样点用于可视化
        current_sampling_points = new_samples.detach().cpu().numpy()
        sampling_points_history.append((f"Adaptive Sampling Iteration {iteration+1}", current_sampling_points))
        
        # 提取新的配点
        t_colloc_new = new_samples[:, 0:1].clone().detach().requires_grad_(True)
        x_colloc_new = new_samples[:, 1:2].clone().detach().requires_grad_(True)
        y_colloc_new = new_samples[:, 2:3].clone().detach().requires_grad_(True)
        
        print(f"获得新的自适应采样点: {new_samples.shape[0]}个")
        
        # 更新配点（保持初始条件和边界条件不变）
        collocation_pts = (t_colloc_new, x_colloc_new, y_colloc_new)
        
        # 使用新采样点继续训练
        print(f"使用新采样点继续训练 ({adaptive_epochs} epochs)...")
        
        model.train()

        for epoch in range(adaptive_epochs):
            optimizer.zero_grad()
            loss, pde_loss, ic_loss, bc_loss = loss_function(model, collocation_pts, initial_pts, boundary_pts,
                                                             lam_u, lam_ut, lam_r)
            
            loss.backward()
            optimizer.step()
            loss_history.append((loss.item(), pde_loss.item(), ic_loss.item(), bc_loss.item()))
            
            #if (epoch + 1) % 200 == 0:
             #   print(f"Epoch [{epoch+1}/{adaptive_epochs}], Loss: {loss.item():.6e}")
            
            if (epoch + 1) % 200 == 0:
                 print(f"Epoch [{epoch+1}/{adaptive_epochs}], "
                       f"Loss: {loss.item():.6e}, "
                       f"PDE Loss: {pde_loss.item():.6e}, "
                       f"IC Loss: {ic_loss.item():.6e}, "
                       f"BC Loss: {bc_loss.item():.6e}")
                 
                 print(f"[NTK] Epoch {epoch+1}: lam_r={lam_r:.4f},lam_ut={lam_ut:.4f}, lam_u={lam_u:.4f}")
            
            
            
            if log_NTK and epoch % 100 == 0:
                print('Computing NTK in retraining round...')
                model.eval()
                D = kernel_size

                # Fresh samples for NTK

                t_ic, x_ic, y_ic, _ = initial_pts
                txy_ic = torch.cat([t_ic.detach(), x_ic.detach(), y_ic.detach()], dim=1)
      
                 # 提取所有 boundary_pts 的 (t, x, y) 并合并
                txy_bc_list = []
                for _, (t_b, x_b, y_b, _) in boundary_pts.items():
                    txy_bc_list.append(torch.cat([t_b.detach(), x_b.detach(), y_b.detach()], dim=1))
                txy_bc = torch.cat(txy_bc_list, dim=0)

                # 合并 initial + boundary
                X_bc_ntk_all = torch.cat([txy_ic, txy_bc], dim=0)  # [N_ic + N_bc_total, 3]

                # 随机取 D 点
                N_bc_total = X_bc_ntk_all.shape[0]
                idx_bc = torch.randperm(N_bc_total)[:D]
                X_bc_ntk = X_bc_ntk_all[idx_bc]  # [D, 3]

                uv_ntk_pred = model(X_bc_ntk)

                u_ntk_pred = uv_ntk_pred[:, [0]] 

                # X_ut_ntk: 从 initial_pts 中随机抽取 D 个点，用于 u_t 损失的 NTK
                t_init_d, x_init_d, y_init_d, _ = initial_pts
                N_init = t_init_d.shape[0]
                idx_ut = torch.randperm(N_init)[:D]
                t_ut = t_init_d[idx_ut].clone().detach().requires_grad_(True)
                x_ut = x_init_d[idx_ut].clone().detach().requires_grad_(True)
                y_ut = y_init_d[idx_ut].clone().detach().requires_grad_(True)

                # X_r_ntk: 从 collocation_pts 中随机抽取 D 个点，用于 PDE residual 的 NTK
                t_c, x_c, y_c = collocation_pts
                N_coll = t_c.shape[0]
                idx_r = torch.randperm(N_coll)[:D]
                t_r = t_c[idx_r].clone().detach().requires_grad_(True)
                x_r = x_c[idx_r].clone().detach().requires_grad_(True)
                y_r = y_c[idx_r].clone().detach().requires_grad_(True)

                # 计算 u_t 预测（初始条件处）
                inputs_ut = torch.cat([t_ut, x_ut, y_ut], dim=1)
                uv_ut = model(inputs_ut)
                u_t_pred = uv_ut[:, [1]]  # v 代表 u_t 的预测值

                 # 计算 PDE residual 预测
                r_pred_ntk = wave_pde_f(model, t_r, x_r, y_r)  # 返回残差向量

                 # 计算 Jacobian 和 NTK
                J_u  = compute_jacobian(model, u_ntk_pred)
                J_ut = compute_jacobian(model, u_t_pred)
                J_r  = compute_jacobian(model, r_pred_ntk)

                K_u  = compute_ntk(J_u,  J_u)
                K_ut = compute_ntk(J_ut, J_ut)
                K_r  = compute_ntk(J_r,  J_r)


                K_u_log.append(K_u)
                K_ut_log.append(K_ut)
                K_r_log.append(K_r)

                if update_lam:
                    trace_K  = np.trace(K_u) + np.trace(K_ut) + np.trace(K_r)
                    lam_u  = min(trace_K / np.trace(K_u), 500)   #trace_K / np.trace(K_u)
                    lam_ut = trace_K / np.trace(K_ut)
                    lam_r  = trace_K / np.trace(K_r)
                    lam_u_log.append(lam_u)
                    lam_ut_log.append(lam_ut)
                    lam_r_log.append(lam_r)
                

                model.train()   

        print(f"迭代 {iteration+1} 完成，当前损失: {loss.item():.6e}")
    
    print("\n" + "=" * 80)
    print("迭代训练完成！")
    print("=" * 80)

                   
    return model, collocation_pts, loss_history, lam_u_log, lam_ut_log, lam_r_log, K_u_log, K_ut_log, K_r_log


In [ ]:
def run_emmpde_experiment(tag, log_ntk_value, update_lam_value):
    global log_NTK, update_lam
    log_NTK = log_ntk_value
    update_lam = update_lam_value
    print('=' * 80)
    print(f'EMMPDE experiment: {tag} (log_NTK={log_NTK}, update_lam={update_lam})')
    print('=' * 80)
    model, collocation_pts, loss_history, lam_u_log, lam_ut_log, lam_r_log, K_u_log, K_ut_log, K_r_log = iterative_training_with_adaptive_sampling_updated(
        initial_epochs=num_epochs,
        adaptive_epochs=num_epochs,
        num_iterations=1,
        mmpde_training_epochs=2000,
        lam_u=1,
        lam_ut=0.1,
        lam_r=0.1
    )
    return {
        'tag': tag,
        'mesh_tag': f'mesh_{nx}_{ny}_{m}_{n}',
        'model': model,
        'collocation_pts': collocation_pts,
        'loss_history': loss_history,
        'lam_u_log': lam_u_log,
        'lam_ut_log': lam_ut_log,
        'lam_r_log': lam_r_log,
        'K_u_log': K_u_log,
        'K_ut_log': K_ut_log,
        'K_r_log': K_r_log,
    }

def compute_global_l2_error(model):
    n_test_points = 8000
    n_per_dim = int(np.cbrt(n_test_points))
    t_test = torch.linspace(0, T, n_per_dim)
    x_test = torch.linspace(0, Lx, n_per_dim)
    y_test = torch.linspace(0, Ly, n_per_dim)
    T_test, X_test, Y_test = torch.meshgrid(t_test, x_test, y_test, indexing='ij')
    t_flat = T_test.reshape(-1, 1)
    x_flat = X_test.reshape(-1, 1)
    y_flat = Y_test.reshape(-1, 1)
    txy_test = torch.cat([t_flat, x_flat, y_flat], dim=1).to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(txy_test)
        u_pred = outputs[:, 0:1]
    u_exact = exact_solution_u(x_flat.cpu().numpy(), y_flat.cpu().numpy(), t_flat.cpu().numpy(), c=c, m_val=m, n_val=n)
    u_exact_tensor = torch.tensor(u_exact, dtype=torch.float32).to(device)
    return torch.sqrt(torch.mean((u_pred - u_exact_tensor)**2)).item()


experiment_results = {}


In [ ]:
current_mesh_tag = set_mesh_case(26, 26, 3, 1)
result = run_emmpde_experiment(f'{current_mesh_tag}_ntk_on', True, True)
experiment_results[f'{current_mesh_tag}_ntk_on'] = result
trained_model = result['model']
collocation_pts = result['collocation_pts']
loss_history = result['loss_history']
lam_u_log = result['lam_u_log']
lam_ut_log = result['lam_ut_log']
lam_r_log = result['lam_r_log']
K_u_log = result['K_u_log']
K_ut_log = result['K_ut_log']
K_r_log = result['K_r_log']
final_l2_ntk_on = compute_global_l2_error(result['model'])
print(f'EMMPDE {current_mesh_tag} ntk_on final L2 error: {final_l2_ntk_on:.6e}')


In [ ]:
current_mesh_tag = set_mesh_case(26, 26, 3, 1)
result = run_emmpde_experiment(f'{current_mesh_tag}_ntk_off', False, False)
experiment_results[f'{current_mesh_tag}_ntk_off'] = result
trained_model_ntk_off = result['model']
collocation_pts_ntk_off = result['collocation_pts']
loss_history_ntk_off = result['loss_history']
final_l2_ntk_off = compute_global_l2_error(result['model'])
print(f'EMMPDE {current_mesh_tag} ntk_off final L2 error: {final_l2_ntk_off:.6e}')


In [ ]:
current_mesh_tag = set_mesh_case(34, 34, 3, 1)
result = run_emmpde_experiment(f'{current_mesh_tag}_ntk_on', True, True)
experiment_results[f'{current_mesh_tag}_ntk_on'] = result
trained_model = result['model']
collocation_pts = result['collocation_pts']
loss_history = result['loss_history']
lam_u_log = result['lam_u_log']
lam_ut_log = result['lam_ut_log']
lam_r_log = result['lam_r_log']
K_u_log = result['K_u_log']
K_ut_log = result['K_ut_log']
K_r_log = result['K_r_log']
final_l2_ntk_on = compute_global_l2_error(result['model'])
print(f'EMMPDE {current_mesh_tag} ntk_on final L2 error: {final_l2_ntk_on:.6e}')


In [ ]:
current_mesh_tag = set_mesh_case(34, 34, 3, 1)
result = run_emmpde_experiment(f'{current_mesh_tag}_ntk_off', False, False)
experiment_results[f'{current_mesh_tag}_ntk_off'] = result
trained_model_ntk_off = result['model']
collocation_pts_ntk_off = result['collocation_pts']
loss_history_ntk_off = result['loss_history']
final_l2_ntk_off = compute_global_l2_error(result['model'])
print(f'EMMPDE {current_mesh_tag} ntk_off final L2 error: {final_l2_ntk_off:.6e}')


In [ ]:
current_mesh_tag = set_mesh_case(51, 51, 3, 1)
result = run_emmpde_experiment(f'{current_mesh_tag}_ntk_on', True, True)
experiment_results[f'{current_mesh_tag}_ntk_on'] = result
trained_model = result['model']
collocation_pts = result['collocation_pts']
loss_history = result['loss_history']
lam_u_log = result['lam_u_log']
lam_ut_log = result['lam_ut_log']
lam_r_log = result['lam_r_log']
K_u_log = result['K_u_log']
K_ut_log = result['K_ut_log']
K_r_log = result['K_r_log']
final_l2_ntk_on = compute_global_l2_error(result['model'])
print(f'EMMPDE {current_mesh_tag} ntk_on final L2 error: {final_l2_ntk_on:.6e}')


In [ ]:
current_mesh_tag = set_mesh_case(51, 51, 3, 1)
result = run_emmpde_experiment(f'{current_mesh_tag}_ntk_off', False, False)
experiment_results[f'{current_mesh_tag}_ntk_off'] = result
trained_model_ntk_off = result['model']
collocation_pts_ntk_off = result['collocation_pts']
loss_history_ntk_off = result['loss_history']
final_l2_ntk_off = compute_global_l2_error(result['model'])
print(f'EMMPDE {current_mesh_tag} ntk_off final L2 error: {final_l2_ntk_off:.6e}')


In [ ]:
def make_tensor(a, requires_grad=False):
    return torch.tensor(a, dtype=torch.float32, device=device, requires_grad=requires_grad)

def evaluate_solution_fields(model, t_value, num_points=101, eps=1e-12):
    xv = np.linspace(0, Lx, num_points)
    yv = np.linspace(0, Ly, num_points)
    X, Y = np.meshgrid(xv, yv, indexing='xy')
    inputs = make_tensor(np.column_stack([np.full(X.size, t_value), X.ravel(), Y.ravel()]))
    model.eval()
    with torch.no_grad():
        prediction = model(inputs)[:, 0].detach().cpu().numpy().reshape(X.shape)
    exact = exact_solution_u(X, Y, t_value, c=c, m_val=m, n_val=n)
    abs_error = np.abs(prediction - exact)
    log_abs_error = np.log10(abs_error + eps)
    return X, Y, prediction, exact, log_abs_error


def plot_final_solution_triplet(model, tag, num_points=101):
    X, Y, prediction, exact, log_abs_error = evaluate_solution_fields(model, T, num_points=num_points)
    vmin_val = min(prediction.min(), exact.min())
    vmax_val = max(prediction.max(), exact.max())
    for field, kwargs in [
        (exact, {'vmin': vmin_val, 'vmax': vmax_val}),
        (prediction, {'vmin': vmin_val, 'vmax': vmax_val}),
        (log_abs_error, {}),
    ]:
        plt.figure()
        image = plt.pcolormesh(X, Y, field, cmap='jet', shading='auto', **kwargs)
        plt.xlabel('$x_1$', fontsize=20)
        plt.ylabel('$x_2$', fontsize=20)
        plt.xlim(0, Lx)
        plt.ylim(0, Ly)
        plt.gca().set_aspect('equal')
        plt.colorbar(image)
        plt.tight_layout()
        plt.show()


def compute_energy_at_time_2d(model, t_value, num_points=101):
    x_grid = np.linspace(0, Lx, num_points)
    y_grid = np.linspace(0, Ly, num_points)
    X, Y = np.meshgrid(x_grid, y_grid, indexing='ij')
    points = np.column_stack([np.full(X.size, t_value), X.ravel(), Y.ravel()])
    points_tensor = make_tensor(points, requires_grad=True)
    model.eval()
    uv = model(points_tensor)
    u = uv[:, [0]]
    u_t = uv[:, 1]
    grad_u = torch.autograd.grad(
        u, points_tensor,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]
    u_x = grad_u[:, 1]
    u_y = grad_u[:, 2]
    energy_density = 0.5 * (u_t**2 + c**2 * (u_x**2 + u_y**2))
    energy_density = energy_density.reshape(num_points, num_points)
    dx = (Lx - 0.0) / (num_points - 1)
    dy = (Ly - 0.0) / (num_points - 1)
    total_energy = torch.trapz(torch.trapz(energy_density, dx=dy, dim=1), dx=dx, dim=0)
    return total_energy.detach().cpu().numpy().item()


def compute_energy_evolution_2d(model, t_array, num_points=101):
    energies = []
    for i, t_val in enumerate(t_array):
        energy = compute_energy_at_time_2d(model, t_val, num_points=num_points)
        energies.append(energy)
        if i % 10 == 0:
            print(f'Energy progress: {i + 1}/{len(t_array)}, t={t_val:.4f}, E={energy:.6e}')
    energies = np.array(energies)
    initial_energy = energies[0]
    relative_errors = np.abs(energies - initial_energy) / (initial_energy + 1e-12)
    return energies, relative_errors, initial_energy


def save_ntk_on_outputs(method_name, result):
    loss_values = np.array(result['loss_history'], dtype=float)
    loss_for_plot = np.column_stack([np.arange(1, len(loss_values) + 1), loss_values[:, 0]])
    np.save(f'loss_history_{method_name}_ntk_on.npy', loss_for_plot)
    print(f"Saved loss history to loss_history_{method_name}_ntk_on.npy")

    t_energy = np.linspace(0, T, 50)
    energies, relative_errors, initial_energy = compute_energy_evolution_2d(result['model'], t_energy)
    np.savez(
        f'{method_name}_energy_data_ntk_on.npz',
        t_array=t_energy,
        time=t_energy,
        energies=energies,
        relative_errors=relative_errors,
        initial_energy=initial_energy,
        method_name=method_name,
        method=method_name,
    )
    print(f"Saved energy data to {method_name}_energy_data_ntk_on.npz")

def plot_sampling_points_ntk_on(result):
    time_values = [0.0, T / 2, T]
    t_coll, x_coll, y_coll = result['collocation_pts']
    t_coll_np = t_coll.detach().cpu().numpy()
    x_coll_np = x_coll.detach().cpu().numpy()
    y_coll_np = y_coll.detach().cpu().numpy()
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for time_idx, t_val in enumerate(time_values):
        time_mask = np.abs(t_coll_np.flatten() - t_val) < 1e-6
        if np.sum(time_mask) > 10:
            axes[time_idx].scatter(x_coll_np[time_mask], y_coll_np[time_mask], c='blue', s=10, alpha=0.6)
            print(f't={t_val:.4f}: displayed {np.sum(time_mask)} sampling points')
        else:
            axes[time_idx].text(0.5, 0.5, f'No points\nnear t={t_val:.4f}',
                                ha='center', va='center', transform=axes[time_idx].transAxes)
            print(f't={t_val:.4f}: not enough sampling points ({np.sum(time_mask)})')
        axes[time_idx].set_xlim(0, Lx)
        axes[time_idx].set_ylim(0, Ly)
        axes[time_idx].set_xlabel('$x_1$', fontsize=20)
        axes[time_idx].set_ylabel('$x_2$', fontsize=20)
        axes[time_idx].grid(True, alpha=0.5, linestyle='-', linewidth=0.5)
        axes[time_idx].set_aspect('equal')
    plt.tight_layout()
    plt.show()


for tag, result in experiment_results.items():
    print(f"Plotting EMMPDE {tag} final-time fields")
    plot_final_solution_triplet(result['model'], tag)

ntk_on_results = {tag: result for tag, result in experiment_results.items() if tag.endswith('_ntk_on')}
if ntk_on_results:
    for tag, result in ntk_on_results.items():
        print(f"Plotting EMMPDE {tag} sampling points")
        plot_sampling_points_ntk_on(result)
        save_ntk_on_outputs(f"EMMPDE_{result['mesh_tag']}", result)

    print('\n' + '=' * 50)
    for tag, result in ntk_on_results.items():
        l2_error = compute_global_l2_error(result['model'])
        print(f"EMMPDE {tag} final L2 error: {l2_error:.6e}")
    print('=' * 50)
else:
    print('Skipping ntk_on output save because experiment_results does not contain ntk_on results.')
